# LunarAI Patch Generation Pipeline

**Project:** Multi-Modal Lunar Image Correspondence & Registration  
**SIH 2026 – ISRO PS26166**  
**Objective:** Generate standardized lunar image patches for LunaDNA training

## Sensors Covered:
- **OHRC** (Orbiter High Resolution Camera)
- **TMC** (Terrain Mapping Camera)
- **IIRS** (Imaging IR Spectrometer)
- **LRO** (Lunar Reconnaissance Orbiter)
- **KAGUYA** (SELENE)

## Pipeline Overview:
1. Load valid images from dataset inventory
2. Read image metadata and convert to grayscale if required
3. Generate overlapping patches (256×256, 25% overlap, 192 stride)
4. Filter invalid patches (empty, dark, bright, low texture, corrupted)
5. Compute patch statistics (mean, std, entropy, texture score)
6. Assign systematic patch IDs
7. Save patch metadata and patches to disk
8. Generate visualizations and summaries

## Configuration:
- **Patch Size:** 256 × 256 pixels
- **Overlap:** 25%
- **Stride:** 192 pixels

## 1. Imports and Configuration

In [ ]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional
from collections import defaultdict
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Data processing
import pandas as pd
import numpy as np

# Image processing
import cv2
from PIL import Image
from skimage import measure
from skimage.feature import graycomatrix, graycoprops
from skimage.filters import sobel

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## 2. Folder Configuration

In [ ]:
# Base paths
BASE_DIR = Path(r"D:\lunar ai")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
METADATA_DIR = BASE_DIR / "metadata"
PATCHES_DIR = BASE_DIR / "patches"
NOTEBOOK_DIR = BASE_DIR / "notebooks"

# Patch configuration
PATCH_SIZE = 256
OVERLAP_RATIO = 0.25  # 25% overlap
STRIDE = int(PATCH_SIZE * (1 - OVERLAP_RATIO))  # 192 pixels

# Quality thresholds for patch filtering
MIN_MEAN_INTENSITY = 10   # Skip very dark patches
MAX_MEAN_INTENSITY = 245  # Skip very bright patches
MIN_STD_DEVIATION = 5    # Skip low texture patches
MIN_ENTROPY = 1.0         # Skip low information patches
MIN_TEXTURE_SCORE = 0.1   # Skip low texture patches

# Create directories if they don't exist
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
METADATA_DIR.mkdir(exist_ok=True, parents=True)
PATCHES_DIR.mkdir(exist_ok=True, parents=True)

print(f"Base Directory: {BASE_DIR}")
print(f"Data Directory: {DATA_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Metadata Directory: {METADATA_DIR}")
print(f"Patches Directory: {PATCHES_DIR}")
print(f"\nPatch Configuration:")
print(f"  Patch Size: {PATCH_SIZE}×{PATCH_SIZE}")
print(f"  Overlap: {OVERLAP_RATIO*100}%")
print(f"  Stride: {STRIDE} pixels")

## 3. Load Dataset Inventory

In [ ]:
def load_dataset_inventory(inventory_path: Path) -> pd.DataFrame:
    """
    Load the dataset inventory from the previous pipeline step.
    
    Args:
        inventory_path: Path to the dataset inventory CSV file
    
    Returns:
        DataFrame containing the dataset inventory
    """
    if not inventory_path.exists():
        raise FileNotFoundError(f"Inventory file not found: {inventory_path}")
    
    print(f"Loading dataset inventory from {inventory_path}")
    df = pd.read_csv(inventory_path)
    print(f"Loaded {len(df)} image records")
    
    # Filter only valid images
    valid_df = df[df['valid'] == True].copy()
    print(f"Valid images: {len(valid_df)}")
    
    return valid_df

## 4. Image Loading and Preprocessing

In [ ]:
def load_image_with_metadata(image_path: str, 
                            convert_grayscale: bool = True) -> Tuple[np.ndarray, Dict]:
    """
    Load an image file and extract metadata.
    
    Args:
        image_path: Path to the image file
        convert_grayscale: Whether to convert to grayscale
    
    Returns:
        Tuple of (image_array, metadata_dict)
    """
    try:
        # Read image based on file type
        if image_path.lower().endswith('.img'):
            # Special handling for IMG files
            img = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
            if img is None:
                raise ValueError(f"Could not read IMG file: {image_path}")
            # Convert BGR to RGB if needed
            if len(img.shape) == 3 and img.shape[2] == 3:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        else:
            # Standard image formats
            with Image.open(image_path) as pil_img:
                img = np.array(pil_img)
                # Convert RGBA to RGB if needed
                if len(img.shape) == 3 and img.shape[2] == 4:
                    img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
        
        # Convert to grayscale if required
        if convert_grayscale:
            if len(img.shape) == 3:
                img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        
        # Normalize to 0-255 range
        if img.dtype != np.uint8:
            img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
        
        # Extract metadata
        metadata = {
            'height': img.shape[0],
            'width': img.shape[1],
            'channels': 1 if len(img.shape) == 2 else img.shape[2],
            'dtype': str(img.dtype),
            'original_path': image_path
        }
        
        return img, metadata
        
    except Exception as e:
        raise ValueError(f"Error loading image {image_path}: {str(e)}")

## 5. Patch Generation Functions

In [ ]:
def generate_patches_from_image(image: np.ndarray,
                               patch_size: int = PATCH_SIZE,
                               stride: int = STRIDE) -> List[Tuple[int, int, np.ndarray]]:
    """
    Generate overlapping patches from an image.
    
    Args:
        image: Input image array
        patch_size: Size of square patches
        stride: Stride for patch extraction
    
    Returns:
        List of tuples (x, y, patch_array)
    """
    patches = []
    height, width = image.shape[:2]
    
    # Calculate number of patches
    n_patches_x = (width - patch_size) // stride + 1
    n_patches_y = (height - patch_size) // stride + 1
    
    # Extract patches
    for y in range(0, height - patch_size + 1, stride):
        for x in range(0, width - patch_size + 1, stride):
            patch = image[y:y+patch_size, x:x+patch_size]
            patches.append((x, y, patch))
    
    return patches

## 6. Patch Quality Assessment

In [ ]:
def compute_patch_statistics(patch: np.ndarray) -> Dict:
    """
    Compute comprehensive statistics for a patch.
    
    Args:
        patch: Patch image array
    
    Returns:
        Dictionary containing patch statistics
    """
    stats = {}
    
    # Basic intensity statistics
    stats['mean_intensity'] = float(np.mean(patch))
    stats['std_deviation'] = float(np.std(patch))
    stats['min_intensity'] = float(np.min(patch))
    stats['max_intensity'] = float(np.max(patch))
    stats['median_intensity'] = float(np.median(patch))
    
    # Entropy (information content)
    hist, _ = np.histogram(patch.flatten(), bins=256, range=(0, 256))
    hist = hist / np.sum(hist)  # Normalize
    hist = hist[hist > 0]  # Remove zero probabilities
    stats['entropy'] = float(-np.sum(hist * np.log2(hist))) if len(hist) > 0 else 0.0
    
    # Texture score using gradient magnitude
    if len(patch.shape) == 2:
        grad_x = cv2.Sobel(patch, cv2.CV_64F, 1, 0, ksize=3)
        grad_y = cv2.Sobel(patch, cv2.CV_64F, 0, 1, ksize=3)
        gradient_magnitude = np.sqrt(grad_x**2 + grad_y**2)
        stats['texture_score'] = float(np.mean(gradient_magnitude))
    else:
        stats['texture_score'] = 0.0
    
    # Edge density
    edges = cv2.Canny(patch, 50, 150)
    stats['edge_density'] = float(np.sum(edges > 0) / edges.size)
    
    return stats

In [ ]:
def is_valid_patch(patch: np.ndarray, 
                   stats: Dict,
                   min_mean: float = MIN_MEAN_INTENSITY,
                   max_mean: float = MAX_MEAN_INTENSITY,
                   min_std: float = MIN_STD_DEVIATION,
                   min_entropy: float = MIN_ENTROPY,
                   min_texture: float = MIN_TEXTURE_SCORE) -> Tuple[bool, str]:
    """
    Determine if a patch meets quality criteria.
    
    Args:
        patch: Patch image array
        stats: Dictionary of patch statistics
        min_mean: Minimum mean intensity threshold
        max_mean: Maximum mean intensity threshold
        min_std: Minimum standard deviation threshold
        min_entropy: Minimum entropy threshold
        min_texture: Minimum texture score threshold
    
    Returns:
        Tuple of (is_valid, rejection_reason)
    """
    # Check for empty or corrupted patches
    if patch is None or patch.size == 0:
        return False, "Empty patch"
    
    # Check for NaN or Inf values
    if np.any(np.isnan(patch)) or np.any(np.isinf(patch)):
        return False, "Contains NaN/Inf values"
    
    # Check intensity thresholds
    if stats['mean_intensity'] < min_mean:
        return False, f"Too dark (mean: {stats['mean_intensity']:.1f})"
    
    if stats['mean_intensity'] > max_mean:
        return False, f"Too bright (mean: {stats['mean_intensity']:.1f})"
    
    # Check texture thresholds
    if stats['std_deviation'] < min_std:
        return False, f"Low contrast (std: {stats['std_deviation']:.1f})"
    
    if stats['entropy'] < min_entropy:
        return False, f"Low entropy (entropy: {stats['entropy']:.2f})"
    
    if stats['texture_score'] < min_texture:
        return False, f"Low texture (score: {stats['texture_score']:.3f})"
    
    return True, "Valid"

## 7. Patch Processing Pipeline

In [ ]:
def process_image_for_patches(image_info: Dict,
                              patch_counter: Dict[str, int],
                              patches_dir: Path) -> List[Dict]:
    """
    Process a single image to generate and save patches.
    
    Args:
        image_info: Dictionary containing image metadata
        patch_counter: Dictionary to track patch IDs per sensor
        patches_dir: Directory to save patches
    
    Returns:
        List of patch metadata dictionaries
    """
    patch_metadata_list = []
    
    try:
        # Load image
        image, img_metadata = load_image_with_metadata(image_info['path'])
        
        # Skip if image is too small
        if image.shape[0] < PATCH_SIZE or image.shape[1] < PATCH_SIZE:
            print(f"  Skipping {image_info['filename']}: Image too small")
            return patch_metadata_list
        
        # Generate patches
        patches = generate_patches_from_image(image)
        
        # Create sensor-specific directory
        sensor_dir = patches_dir / image_info['sensor']
        sensor_dir.mkdir(exist_ok=True, parents=True)
        
        # Process each patch
        valid_count = 0
        rejected_count = 0
        
        for x, y, patch in patches:
            # Compute statistics
            stats = compute_patch_statistics(patch)
            
            # Validate patch
            is_valid, rejection_reason = is_valid_patch(patch, stats)
            
            if is_valid:
                # Generate patch ID
                sensor = image_info['sensor']
                patch_counter[sensor] = patch_counter.get(sensor, 0) + 1
                patch_id = f"PATCH_{patch_counter[sensor]:06d}"
                
                # Save patch
                patch_filename = f"{patch_id}.png"
                patch_path = sensor_dir / patch_filename
                cv2.imwrite(str(patch_path), patch)
                
                # Create metadata entry
                patch_metadata = {
                    'patch_id': patch_id,
                    'source_image': image_info['filename'],
                    'sensor': sensor,
                    'dataset': image_info['dataset'],
                    'x_coord': x,
                    'y_coord': y,
                    'patch_size': PATCH_SIZE,
                    'patch_path': str(patch_path),
                    **stats
                }
                patch_metadata_list.append(patch_metadata)
                valid_count += 1
            else:
                rejected_count += 1
        
        print(f"  Generated {valid_count} valid patches, {rejected_count} rejected")
        
    except Exception as e:
        print(f"  Error processing {image_info['filename']}: {str(e)}")
    
    return patch_metadata_list

## 8. Main Patch Generation Pipeline

In [ ]:
def generate_patches_pipeline(inventory_df: pd.DataFrame,
                              patches_dir: Path) -> pd.DataFrame:
    """
    Main pipeline to generate patches from all valid images.
    
    Args:
        inventory_df: DataFrame containing valid image inventory
        patches_dir: Directory to save patches
    
    Returns:
        DataFrame containing all patch metadata
    """
    print("="*60)
    print("PATCH GENERATION PIPELINE")
    print("="*60)
    
    all_patch_metadata = []
    patch_counter = defaultdict(int)
    
    # Group images by sensor for organized processing
    sensor_groups = inventory_df.groupby('sensor')
    
    for sensor, sensor_df in sensor_groups:
        print(f"\nProcessing {sensor} images ({len(sensor_df)} images)...")
        print("-" * 50)
        
        for _, image_info in tqdm(sensor_df.iterrows(), total=len(sensor_df), desc=f"{sensor}"):
            image_dict = image_info.to_dict()
            patch_metadata = process_image_for_patches(image_dict, patch_counter, patches_dir)
            all_patch_metadata.extend(patch_metadata)
    
    # Create metadata DataFrame
    if all_patch_metadata:
        patch_df = pd.DataFrame(all_patch_metadata)
        
        # Reorder columns for better readability
        columns = ['patch_id', 'source_image', 'sensor', 'dataset', 'x_coord', 'y_coord',
                  'patch_size', 'patch_path', 'mean_intensity', 'std_deviation',
                  'min_intensity', 'max_intensity', 'median_intensity', 'entropy',
                  'texture_score', 'edge_density']
        patch_df = patch_df.reindex(columns=columns)
        
        print(f"\n{'='*60}")
        print(f"Total patches generated: {len(patch_df)}")
        print(f"{'='*60}")
    else:
        patch_df = pd.DataFrame()
        print("\nNo patches were generated!")
    
    return patch_df

## 9. Patch Statistics Generation

In [ ]:
def generate_patch_statistics(patch_df: pd.DataFrame) -> pd.DataFrame:
    """
    Generate comprehensive statistics for the patch dataset.
    
    Args:
        patch_df: DataFrame containing patch metadata
    
    Returns:
        DataFrame containing patch statistics
    """
    print("Generating patch statistics...")
    
    if patch_df.empty:
        print("No patch data available for statistics")
        return pd.DataFrame()
    
    # Overall statistics
    total_patches = len(patch_df)
    total_sensors = patch_df['sensor'].nunique()
    total_images = patch_df['source_image'].nunique()
    
    # Statistics by sensor
    sensor_stats = patch_df.groupby('sensor').agg({
        'patch_id': 'count',
        'source_image': 'nunique',
        'mean_intensity': ['mean', 'std', 'min', 'max'],
        'std_deviation': ['mean', 'std'],
        'entropy': ['mean', 'std'],
        'texture_score': ['mean', 'std']
    }).round(3)
    
    sensor_stats.columns = ['patch_count', 'image_count',
                           'mean_intensity_mean', 'mean_intensity_std',
                           'mean_intensity_min', 'mean_intensity_max',
                           'std_deviation_mean', 'std_deviation_std',
                           'entropy_mean', 'entropy_std',
                           'texture_score_mean', 'texture_score_std']
    
    # Create overall statistics DataFrame
    overall_stats = pd.DataFrame({
        'metric': [
            'total_patches',
            'total_sensors',
            'total_source_images',
            'avg_patches_per_image',
            'avg_mean_intensity',
            'avg_std_deviation',
            'avg_entropy',
            'avg_texture_score'
        ],
        'value': [
            total_patches,
            total_sensors,
            total_images,
            round(total_patches / total_images, 2) if total_images > 0 else 0,
            round(patch_df['mean_intensity'].mean(), 2),
            round(patch_df['std_deviation'].mean(), 2),
            round(patch_df['entropy'].mean(), 3),
            round(patch_df['texture_score'].mean(), 3)
        ]
    })
    
    print("\n=== Overall Patch Statistics ===")
    print(overall_stats.to_string(index=False))
    
    print("\n=== Statistics by Sensor ===")
    print(sensor_stats.to_string())
    
    return overall_stats, sensor_stats

## 10. Visualization Functions

In [ ]:
def visualize_patch_distribution(patch_df: pd.DataFrame, output_path: Path):
    """
    Create a bar chart showing patch distribution by sensor.
    
    Args:
        patch_df: DataFrame containing patch metadata
        output_path: Path to save the plot
    """
    if patch_df.empty:
        print("No patch data available for visualization")
        return
    
    sensor_counts = patch_df['sensor'].value_counts().sort_values(ascending=True)
    
    plt.figure(figsize=(10, 6))
    colors = plt.cm.Set3(range(len(sensor_counts)))
    sensor_counts.plot(kind='barh', color=colors)
    plt.title('Patch Distribution by Sensor', fontsize=14, fontweight='bold')
    plt.xlabel('Number of Patches', fontsize=12)
    plt.ylabel('Sensor', fontsize=12)
    plt.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(sensor_counts):
        plt.text(v + 0.5, i, str(v), va='center', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {output_path}")

In [ ]:
def visualize_texture_distribution(patch_df: pd.DataFrame, output_path: Path):
    """
    Create a histogram showing texture score distribution.
    
    Args:
        patch_df: DataFrame containing patch metadata
        output_path: Path to save the plot
    """
    if patch_df.empty:
        print("No patch data available for visualization")
        return
    
    plt.figure(figsize=(12, 6))
    
    # Overall texture distribution
    plt.subplot(1, 2, 1)
    plt.hist(patch_df['texture_score'], bins=50, color='steelblue', 
             edgecolor='black', alpha=0.7)
    plt.title('Overall Texture Score Distribution', fontsize=12, fontweight='bold')
    plt.xlabel('Texture Score', fontsize=10)
    plt.ylabel('Frequency', fontsize=10)
    plt.grid(axis='y', alpha=0.3)
    
    # Texture distribution by sensor
    plt.subplot(1, 2, 2)
    sensors = patch_df['sensor'].unique()
    for i, sensor in enumerate(sensors):
        sensor_data = patch_df[patch_df['sensor'] == sensor]['texture_score']
        plt.hist(sensor_data, bins=30, alpha=0.5, label=sensor, 
                color=plt.cm.Set3(i))
    
    plt.title('Texture Score by Sensor', fontsize=12, fontweight='bold')
    plt.xlabel('Texture Score', fontsize=10)
    plt.ylabel('Frequency', fontsize=10)
    plt.legend(fontsize=8)
    plt.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {output_path}")

In [ ]:
def visualize_sample_patches(patch_df: pd.DataFrame, 
                           patches_dir: Path,
                           output_path: Path,
                           samples_per_sensor: int = 8):
    """
    Create a grid visualization of sample patches.
    
    Args:
        patch_df: DataFrame containing patch metadata
        patches_dir: Directory containing patches
        output_path: Path to save the plot
        samples_per_sensor: Number of sample patches per sensor
    """
    if patch_df.empty:
        print("No patch data available for visualization")
        return
    
    sensors = patch_df['sensor'].unique()
    n_sensors = len(sensors)
    
    # Calculate grid dimensions
    n_cols = min(4, samples_per_sensor)
    n_rows_per_sensor = (samples_per_sensor + n_cols - 1) // n_cols
    n_rows = n_rows_per_sensor * n_sensors
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
    if n_sensors == 1 and n_rows == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    plot_idx = 0
    
    for sensor in sensors:
        sensor_patches = patch_df[patch_df['sensor'] == sensor].head(samples_per_sensor)
        
        for _, patch_info in sensor_patches.iterrows():
            if plot_idx >= len(axes):
                break
            
            try:
                patch_path = Path(patch_info['patch_path'])
                if patch_path.exists():
                    patch_img = cv2.imread(str(patch_path), cv2.IMREAD_GRAYSCALE)
                    axes[plot_idx].imshow(patch_img, cmap='gray')
                    axes[plot_idx].set_title(f"{sensor}\n{patch_info['patch_id']}", fontsize=8)
                    axes[plot_idx].axis('off')
                else:
                    axes[plot_idx].text(0.5, 0.5, 'Patch not found', 
                                      ha='center', va='center', transform=axes[plot_idx].transAxes)
                    axes[plot_idx].set_title(f"{sensor}\n{patch_info['patch_id']}", fontsize=8)
                    axes[plot_idx].axis('off')
            except Exception as e:
                axes[plot_idx].text(0.5, 0.5, f'Error: {str(e)[:20]}', 
                                  ha='center', va='center', transform=axes[plot_idx].transAxes)
                axes[plot_idx].set_title(f"{sensor}\n{patch_info['patch_id']}", fontsize=8)
                axes[plot_idx].axis('off')
            
            plot_idx += 1
    
    # Hide unused subplots
    for i in range(plot_idx, len(axes)):
        axes[i].axis('off')
    
    plt.suptitle('Sample Patches by Sensor', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {output_path}")

## 11. Save Outputs

In [ ]:
def save_patch_outputs(patch_df: pd.DataFrame,
                       overall_stats: pd.DataFrame,
                       sensor_stats: pd.DataFrame,
                       metadata_dir: Path,
                       output_dir: Path):
    """
    Save all patch-related outputs to files.
    
    Args:
        patch_df: DataFrame containing patch metadata
        overall_stats: DataFrame containing overall statistics
        sensor_stats: DataFrame containing sensor-specific statistics
        metadata_dir: Directory to save metadata files
        output_dir: Directory to save visualization files
    """
    print("\nSaving outputs...")
    
    # Save patch metadata
    metadata_path = metadata_dir / "patch_metadata.csv"
    patch_df.to_csv(metadata_path, index=False)
    print(f"Saved: {metadata_path}")
    
    # Save overall statistics
    overall_stats_path = metadata_dir / "patch_statistics.csv"
    overall_stats.to_csv(overall_stats_path, index=False)
    print(f"Saved: {overall_stats_path}")
    
    # Save sensor statistics
    sensor_stats_path = metadata_dir / "sensor_patch_statistics.csv"
    sensor_stats.to_csv(sensor_stats_path)
    print(f"Saved: {sensor_stats_path}")
    
    # Save comprehensive summary as JSON
    summary = {
        'timestamp': datetime.now().isoformat(),
        'total_patches': len(patch_df),
        'total_sensors': patch_df['sensor'].nunique(),
        'total_source_images': patch_df['source_image'].nunique(),
        'patch_size': PATCH_SIZE,
        'overlap_ratio': OVERLAP_RATIO,
        'stride': STRIDE,
        'sensors': patch_df['sensor'].unique().tolist(),
        'quality_thresholds': {
            'min_mean_intensity': MIN_MEAN_INTENSITY,
            'max_mean_intensity': MAX_MEAN_INTENSITY,
            'min_std_deviation': MIN_STD_DEVIATION,
            'min_entropy': MIN_ENTROPY,
            'min_texture_score': MIN_TEXTURE_SCORE
        }
    }
    
    summary_path = metadata_dir / "patch_generation_summary.json"
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"Saved: {summary_path}")
    
    print("\nAll outputs saved successfully!")

## 12. Display Final Summary

In [ ]:
def display_patch_summary(patch_df: pd.DataFrame,
                         patches_dir: Path):
    """
    Display a comprehensive summary of the patch generation process.
    
    Args:
        patch_df: DataFrame containing patch metadata
        patches_dir: Directory containing patches
    """
    print("\n" + "="*60)
    print("PATCH GENERATION - FINAL SUMMARY")
    print("="*60)
    
    if patch_df.empty:
        print("No patches were generated.")
        return
    
    total_patches = len(patch_df)
    total_sensors = patch_df['sensor'].nunique()
    total_images = patch_df['source_image'].nunique()
    avg_texture = patch_df['texture_score'].mean()
    
    # Calculate storage usage
    storage_bytes = 0
    for sensor_dir in patches_dir.iterdir():
        if sensor_dir.is_dir():
            for patch_file in sensor_dir.glob("*.png"):
                storage_bytes += patch_file.stat().st_size
    
    storage_mb = storage_bytes / (1024 * 1024)
    storage_gb = storage_bytes / (1024 * 1024 * 1024)
    
    print(f"\n📊 OVERALL STATISTICS")
    print(f"{'─'*40}")
    print(f"Total Images Processed: {total_images}")
    print(f"Total Patches Generated: {total_patches:,}")
    print(f"Total Sensors: {total_sensors}")
    print(f"Average Patches per Image: {total_patches/total_images:.1f}")
    print(f"Storage Used: {storage_gb:.2f} GB ({storage_mb:.2f} MB)")
    print(f"\n🎯 QUALITY METRICS")
    print(f"{'─'*40}")
    print(f"Average Texture Score: {avg_texture:.3f}")
    print(f"Average Mean Intensity: {patch_df['mean_intensity'].mean():.1f}")
    print(f"Average Std Deviation: {patch_df['std_deviation'].mean():.1f}")
    print(f"Average Entropy: {patch_df['entropy'].mean():.3f}")
    
    print(f"\n📡 SENSOR BREAKDOWN")
    print(f"{'─'*40}")
    sensor_counts = patch_df['sensor'].value_counts()
    for sensor, count in sensor_counts.items():
        percentage = count / total_patches * 100
        sensor_patches = patch_df[patch_df['sensor'] == sensor]
        avg_texture_sensor = sensor_patches['texture_score'].mean()
        print(f"{sensor:12s}: {count:6,} patches ({percentage:5.1f}%) - Avg texture: {avg_texture_sensor:.3f}")
    
    print(f"\n" + "="*60)
    print(f"Pipeline completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)

## 13. Main Execution Function

In [ ]:
def main():
    """
    Main execution function for the patch generation pipeline.
    """
    print("="*60)
    print("LUNARAI PATCH GENERATION PIPELINE")
    print("="*60)
    print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print()
    
    try:
        # Step 1: Load dataset inventory
        print("\n" + "="*60)
        print("STEP 1: Loading Dataset Inventory")
        print("="*60)
        inventory_path = OUTPUT_DIR / "dataset_inventory.csv"
        inventory_df = load_dataset_inventory(inventory_path)
        
        if inventory_df.empty:
            print("ERROR: No valid images found in inventory.")
            return
        
        # Step 2: Generate patches
        print("\n" + "="*60)
        print("STEP 2: Generating Patches")
        print("="*60)
        patch_df = generate_patches_pipeline(inventory_df, PATCHES_DIR)
        
        if patch_df.empty:
            print("ERROR: No patches were generated.")
            return
        
        # Step 3: Generate statistics
        print("\n" + "="*60)
        print("STEP 3: Generating Patch Statistics")
        print("="*60)
        overall_stats, sensor_stats = generate_patch_statistics(patch_df)
        
        # Step 4: Create visualizations
        print("\n" + "="*60)
        print("STEP 4: Creating Visualizations")
        print("="*60)
        visualize_patch_distribution(patch_df, OUTPUT_DIR / "patch_distribution.png")
        visualize_texture_distribution(patch_df, OUTPUT_DIR / "texture_distribution.png")
        visualize_sample_patches(patch_df, PATCHES_DIR, OUTPUT_DIR / "sample_patches.png")
        
        # Step 5: Save outputs
        print("\n" + "="*60)
        print("STEP 5: Saving Outputs")
        print("="*60)
        save_patch_outputs(patch_df, overall_stats, sensor_stats, METADATA_DIR, OUTPUT_DIR)
        
        # Step 6: Display final summary
        print("\n" + "="*60)
        print("STEP 6: Final Summary")
        print("="*60)
        display_patch_summary(patch_df, PATCHES_DIR)
        
        print("\n" + "="*60)
        print("✅ PATCH GENERATION COMPLETED SUCCESSFULLY")
        print("="*60)
        
    except Exception as e:
        print(f"\n❌ ERROR: Pipeline failed with exception:")
        print(f"{type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        
    print(f"\nFinished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 14. Execute Pipeline

In [ ]:
# Run the main pipeline
if __name__ == "__main__":
    main()

---

## Pipeline Complete!

This notebook has successfully:

1. ✅ Loaded valid images from dataset inventory
2. ✅ Read image metadata and converted to grayscale
3. ✅ Generated overlapping patches (256×256, 25% overlap, 192 stride)
4. ✅ Filtered invalid patches (empty, dark, bright, low texture, corrupted)
5. ✅ Computed patch statistics (mean, std, entropy, texture score)
6. ✅ Assigned systematic patch IDs (PATCH_000001, PATCH_000002, etc.)
7. ✅ Saved patch metadata and patches to disk
8. ✅ Generated comprehensive visualizations
9. ✅ Created detailed statistics and summaries

### Output Files Generated:
- `patches/[sensor]/PATCH_*.png` - Individual patch images organized by sensor
- `metadata/patch_metadata.csv` - Complete metadata for all patches
- `metadata/patch_statistics.csv` - Overall patch statistics
- `metadata/sensor_patch_statistics.csv` - Sensor-specific statistics
- `metadata/patch_generation_summary.json` - Comprehensive summary in JSON format
- `outputs/patch_distribution.png` - Bar chart of patches by sensor
- `outputs/texture_distribution.png` - Texture score distribution histograms
- `outputs/sample_patches.png` - Grid visualization of sample patches

### Quality Filtering Applied:
- **Minimum Mean Intensity:** 10 (skips very dark patches)
- **Maximum Mean Intensity:** 245 (skips very bright patches)
- **Minimum Standard Deviation:** 5 (skips low contrast patches)
- **Minimum Entropy:** 1.0 (skips low information patches)
- **Minimum Texture Score:** 0.1 (skips low texture patches)

### Next Steps:
- Review the generated patches in the `patches/` directory
- Analyze the statistics and visualizations for quality assessment
- Adjust quality thresholds if needed based on your requirements
- Use the patch metadata for subsequent LunaDNA training pipelines
- Consider data augmentation strategies for training diversity